In [1]:
import joblib
import pandas as pd
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from scipy.stats import uniform, randint
from xgboost import XGBClassifier

In [2]:
import warnings
warnings.filterwarnings('ignore')

In [3]:
df = pd.read_csv('data/train.csv')
df = pd.read_csv('data/train.csv').dropna(subset=['Churn']).drop(columns=['CustomerID'])

X = df.drop(columns=['Churn'])
y = df['Churn']

In [4]:
class FeatureEngineer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X['Tenure'] = X['Tenure'].fillna(X['Tenure'].median())
        X['Total Spend'] = X['Total Spend'].fillna(X['Total Spend'].median())
        X['Support Calls'] = X['Support Calls'].fillna(X['Support Calls'].median())
        X['Usage Frequency'] = X['Usage Frequency'].fillna(X['Usage Frequency'].median())
        X['Payment Delay'] = X['Payment Delay'].fillna(0)
        X['avg_monthly_spend'] = X['Total Spend'] / X['Tenure'].replace(0, 1)
        X['support_freq_ratio'] = X['Support Calls'] / X['Tenure'].replace(0, 1)
        X['usage_per_month'] = X['Usage Frequency'] / X['Tenure'].replace(0, 1)
        X['has_payment_delay'] = (X['Payment Delay'] > 0).astype(int)
        return X

In [5]:
class OutlierCapper(BaseEstimator, TransformerMixin):
    def __init__(self, lower=0.01, upper=0.99):
        self.lower = lower
        self.upper = upper

    def fit(self, X, y=None):
        self.bounds_ = {}
        for i in range(X.shape[1]):
            self.bounds_[i] = (np.quantile(X[:, i], self.lower), np.quantile(X[:, i], self.upper))
        return self

    def transform(self, X):
        X = X.copy()
        for i, (low, high) in self.bounds_.items():
            X[:, i] = np.clip(X[:, i], low, high)
        return X

In [6]:
feature_engineered_X = FeatureEngineer().fit_transform(X)
num_cols = feature_engineered_X.select_dtypes(include='number').columns.tolist()
cat_cols = feature_engineered_X.select_dtypes(include=['object', 'str']).columns.tolist()

preprocessor = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('capper', OutlierCapper()),
        ('scaler', StandardScaler()),
    ]), num_cols),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)),
    ]), cat_cols)
])

In [7]:
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

In [8]:
models = {
    "LR": (LogisticRegression(max_iter=1000), {
        'C': uniform(0.01, 5),
        'penalty': ['l2']
    }),
    "RF": (RandomForestClassifier(), {
        'n_estimators': randint(50, 100),
        'max_depth': randint(3, 8),
        'min_samples_split': randint(2, 5)
    }),
    "XGB": (XGBClassifier(eval_metric='logloss'), {
        'n_estimators': randint(50, 100),
        'max_depth': randint(3, 6),
        'learning_rate': uniform(0.05, 0.2)
    })
}

In [9]:
best_overall_score = 0
best_overall_model = None
best_overall_name = None
results = []

for name, (model, params) in models.items():
    print(f"\n>>> Training {name}...")
    
    full_model = Pipeline([
        ('features', FeatureEngineer()),
        ('preprocess', preprocessor), 
        ('clf', model)
    ])
    
    rs = RandomizedSearchCV(full_model, {'clf__' + k: v for k, v in params.items()}, 
                           n_iter=10, cv=skf, scoring='accuracy', random_state=42, n_jobs=-1)
    rs.fit(X, y)
    
    print(f"    Best CV accuracy: {rs.best_score_:.3f}")
    print(f"    Best params: {rs.best_params_}")
    
    best = rs.best_estimator_
    y_pred = best.predict(X)
    
    acc = accuracy_score(y, y_pred)
    results.append({
        'Model': name,
        'Accuracy': acc,
        'Precision': precision_score(y, y_pred),
        'Recall': recall_score(y, y_pred),
        'F1': f1_score(y, y_pred)
    })
    
    if acc > best_overall_score:
        best_overall_score = acc
        best_overall_model = best
        best_overall_name = name

df_results = pd.DataFrame(results)
print("\n>>> Results:")
print(df_results.round(3))


>>> Training LR...
    Best CV accuracy: 0.852
    Best params: {'clf__C': np.float64(0.3004180608409973), 'clf__penalty': 'l2'}

>>> Training RF...
    Best CV accuracy: 0.965
    Best params: {'clf__max_depth': 7, 'clf__min_samples_split': 2, 'clf__n_estimators': 98}

>>> Training XGB...
    Best CV accuracy: 0.998
    Best params: {'clf__learning_rate': np.float64(0.11084844859190755), 'clf__max_depth': 4, 'clf__n_estimators': 93}

>>> Results:
  Model  Accuracy  Precision  Recall     F1
0    LR     0.852      0.882   0.854  0.868
1    RF     0.960      0.994   0.935  0.964
2   XGB     0.998      1.000   0.996  0.998


In [10]:
joblib.dump(best_overall_model, 'best_churn_model.pkl')
print(f"Saved best model: {best_overall_name} (Accuracy: {best_overall_score:.3f})")

Saved best model: XGB (Accuracy: 0.998)
